# **HANDS - ON: FUNDAMENTOS DE LLMS CON MODELOS LLAMA**


Una vez vista la masterclass ***Fundamentos de LLMs y Arquitectura de Llama***, se proporciona el siguiente ***Colab*** para construir, en vivo, la primera llamada a Llama y medir su comportamiento.

Usamos **Ollama** para ejecutar Llama directamente en la GPU gratuita de Colab, sin necesidad de API key, cuentas externas ni infraestructura propia.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1bjMHw7fmMxo9dyQp140e9mqotKRkfKBg?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

Antes de llamar a Llama necesitamos activar la GPU gratuita de Colab, instalar **Ollama** (el programa que descarga y ejecuta los modelos) y descargar los modelos que vamos a usar. La librería `ollama` es el cliente oficial en Python para hablar con ese servidor.

### **ACTIVAR LA GPU**

En el menú de Colab ve a **Entorno de ejecución → Cambiar tipo de entorno de ejecución** y selecciona **T4 GPU**. Luego ejecuta la celda para confirmar que Colab te asignó la GPU.

In [5]:
# Verificar que Colab asignó una GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


### **INSTALAR OLLAMA**

A diferencia de una API en la nube, aquí **el modelo corre dentro de tu propia sesión de Colab**: no hay *API key* ni *Secrets*. Ollama se instala en la máquina virtual, se arranca como un servidor local (`localhost:11434`) y desde Python le enviamos los prompts.

**IMPORTANTE:** la sesión de Colab es temporal. Si se reinicia o se desconecta, hay que volver a ejecutar las celdas de instalación, servidor y descarga de modelos.

In [6]:
# Instalar Ollama en la máquina virtual de Colab (tarda ~1 minuto)
!sudo apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

print("Ollama instalado.")

Ollama instalado.


In [7]:
# Arrancar el servidor de Ollama en segundo plano y esperar a que responda
import subprocess
import time
import requests

servidor = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):
    try:
        requests.get("http://localhost:11434")
        print("Servidor de Ollama listo.")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    print("El servidor no respondió. Vuelve a ejecutar esta celda.")
!ollama --version

Servidor de Ollama listo.
ollama version is 0.40.0


### **DESCARGAR LOS MODELOS**

Vamos a trabajar con dos tamaños de Llama: **Llama 3.2 de 3B** (ligero) y **Llama 3.1 de 8B** (más grande). Guardamos los nombres en variables para usar *exactamente la misma etiqueta* al descargar y al llamar al modelo; si no coinciden carácter por carácter, Ollama devuelve error.

In [8]:
# Descargar los dos modelos Llama (la primera vez tarda unos minutos)
MODELO_LIGERO = "llama3.2:3b"
MODELO_GRANDE = "llama3.1:8b"

!ollama pull {MODELO_LIGERO}
!ollama pull {MODELO_GRANDE}
!ollama list



NAME           ID              SIZE      MODIFIED               
llama3.1:8b    46e0c10c039e    4.9 GB    Less than a second ago    
llama3.2:3b    a80c4f17acd5    2.0 GB    53 seconds ago            


### **CLIENTE DE PYTHON**

In [9]:
# Instalar el cliente de Ollama y conectarlo al servidor local

!pip install ollama -q

import ollama
import time

client = ollama.Client(host="http://localhost:11434")
print("Cliente de Ollama inicializado correctamente.")

Cliente de Ollama inicializado correctamente.


## **DE PALABRAS A TOKENS**

### **PRIMERA LLAMADA A LLAMA (ZERO-SHOT)**

Un LLM predice el siguiente token más probable, **no busca la respuesta en una base de datos**. Antes de que Llama genere una sola palabra, tu prompt se divide en **tokens**.

Vamos a enviar el prompt a Llama en modo zero-shot (sin ejemplos previos) y ver la respuesta generada.

In [10]:
# Definir y enviar un prompt a Llama en Ollama y mostrar la respuesta generada

prompt = "¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?"

response = client.chat(
    model=MODELO_LIGERO,
    messages=[{"role": "user", "content": prompt}]
)

print(prompt)
print(response.message.content)

¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?
**Diferencia entre RAM y almacenamiento en una computadora**

La RAM (Random Access Memory) y el almacenamiento en una computadora son dos conceptos fundamentales que funcionan de manera diferente para almacenar y recuperar datos.

**RAM (Random Access Memory)**

La RAM es un tipo de memoria que permite acceder rápidamente a los datos almacenados en ella. Es un tipo de memoria volátil, lo que significa que los datos se pierden cuando se apaga la computadora. La RAM es utilizada por el procesador para realizar operaciones de cálculo y almacenar los datos que se están procesando en el momento.

*   La RAM es rápida y eficiente para realizar operaciones de cálculo.
*   La RAM es volátil, lo que significa que los datos se pierden cuando se apaga la computadora.
*   La RAM es utilizada por el procesador para realizar operaciones de cálculo y almacenar los datos que se están procesando en el momento.

**Almacenamient

In [11]:
print(response.model_dump_json(indent=2))  # para inspeccionar la respuesta completa

{
  "model": "llama3.2:3b",
  "created_at": "2026-10-07T20:23:36.611924205Z",
  "done": true,
  "done_reason": "stop",
  "total_duration": 101382186807,
  "load_duration": 54440179183,
  "prompt_eval_count": 43,
  "prompt_eval_duration": 39763404000,
  "eval_count": 600,
  "eval_duration": 7175598000,
  "message": {
    "role": "assistant",
    "content": "**Diferencia entre RAM y almacenamiento en una computadora**\n\nLa RAM (Random Access Memory) y el almacenamiento en una computadora son dos conceptos fundamentales que funcionan de manera diferente para almacenar y recuperar datos.\n\n**RAM (Random Access Memory)**\n\nLa RAM es un tipo de memoria que permite acceder rápidamente a los datos almacenados en ella. Es un tipo de memoria volátil, lo que significa que los datos se pierden cuando se apaga la computadora. La RAM es utilizada por el procesador para realizar operaciones de cálculo y almacenar los datos que se están procesando en el momento.\n\n*   La RAM es rápida y eficiente 

### **CUÁNTOS TOKENS CONSUMIÓ EL PROMPT**

La respuesta de Ollama trae el conteo de tokens de entrada (`prompt_eval_count`) y de salida (`eval_count`). Es la forma más directa de responder: ¿cuántos tokens consumió mi prompt?

In [12]:
# Mostrar cuántos tokens tuvo el prompt y cuántos tuvo la respuesta

tokens_prompt = response.prompt_eval_count
tokens_respuesta = response.eval_count

print("Tokens del prompt:", tokens_prompt)
print("Tokens de la respuesta:", tokens_respuesta)
print("Tokens totales:", tokens_prompt + tokens_respuesta)
# Con Ollama no se factura: los tokens miden cuánto trabajo hizo el modelo y cuánto de su contexto usaste


Tokens del prompt: 43
Tokens de la respuesta: 600
Tokens totales: 643


### **MIDIENDO LA LATENCIA DE INFERENCIA**

Elegir el tamaño correcto de modelo es una decisión de ingeniería, no solo de potencia bruta. Vamos a medir cuánto tarda Llama en responder el mismo prompt.

In [13]:
# Medir el tiempo de respuesta de Llama para el mismo prompt

inicio = time.time()
response_tiempo = client.chat(
    model=MODELO_LIGERO,
    messages=[{"role": "user", "content": prompt}]
)
duracion = time.time() - inicio
# duracion_ms = duracion * 1000  # si prefieres reportarlo en milisegundos

print(f"Tiempo de respuesta: {duracion:.2f} segundos")

Tiempo de respuesta: 8.85 segundos


### **COMPARANDO DOS TAMAÑOS DE MODELO**

Repite la misma llamada usando la versión más grande de Llama que cabe en la GPU gratuita de Colab (8B) y compara tiempo de respuesta y calidad contra el modelo ligero (3B).

Ojo: la primera vez que se usa un modelo, Ollama tiene que **cargarlo en la memoria de la GPU**. Ese tiempo de carga viene en `load_duration` y conviene separarlo para que la comparación sea justa.

In [14]:
# Definir y enviar un prompt a Llama en Ollama y mostrar la respuesta generada

inicio = time.time()
response_grande = client.chat(
    model=MODELO_GRANDE,
    messages=[{"role": "user", "content": prompt}]
)
duracion_grande = time.time() - inicio
carga_grande = response_grande.load_duration / 1e9  # nanosegundos -> segundos

tokens_ligero = response.prompt_eval_count + response.eval_count
tokens_grande = response_grande.prompt_eval_count + response_grande.eval_count

print(f"Modelo ligero ({MODELO_LIGERO}): {duracion:.2f} s — {tokens_ligero} tokens")
print(f"Modelo grande ({MODELO_GRANDE}): {duracion_grande:.2f} s — {tokens_grande} tokens (de los cuales {carga_grande:.2f} s fueron carga del modelo)")
print("\nRespuesta del modelo grande:\n", response_grande.message.content)

Modelo ligero (llama3.2:3b): 8.85 s — 643 tokens
Modelo grande (llama3.1:8b): 19.10 s — 90 tokens (de los cuales 17.21 s fueron carga del modelo)

Respuesta del modelo grande:
 La RAM (Memoria de Acceso Aleatorio) es una memoria de alta velocidad utilizada para almacenar y procesar la información temporal de la computadora. Por otro lado, el almacenamiento es una unidad de almacenamiento en disco que almacena los datos permanentes de la computadora.


## **CHALLENGE: COMPARADOR DE MODELOS LLAMA**

Una vez visto el ***Hands-On: Fundamentos de LLMs***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un pequeño **comparador que envíe 3 preguntas** reales a Llama y registre, para cada una, el **tiempo de respuesta** y el **número de tokens** consumidos.

**IMPORTANTE:** Para su revisión, **es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.**

### **INSTRUCCIONES:**

**1. Carga la API key y genera 3 preguntas:**

   * Lee la API key de llama ya configurada desde **Colab Secrets**.
   * Construye una lista vacía llamada `preguntas` y agrégale 3 preguntas frecuentes de tu propio contexto (soporte técnico, tienda, escuela, etc.).

In [1]:
# Leer API key desde Colab Secrets
# RETO 1. Cargar API Key y definir 3 preguntas

!pip -q install groq

from google.colab import userdata

# Leer la API key guardada en Colab Secrets
GROQ_API_KEY = userdata.get("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("No se encontró GROQ_API_KEY en Colab Secrets.")

# Cliente de Groq
from groq import Groq
client = Groq(api_key=GROQ_API_KEY)

# Lista de tres preguntas de contexto propio
preguntas = []
preguntas.append("¿Cómo puedo mejorar la seguridad de una computadora personal?")
preguntas.append("¿Qué estrategia recomendarías para estudiar matemáticas antes de un examen?")
preguntas.append("¿Cómo puede una escuela utilizar inteligencia artificial de manera responsable?")

print("API Key cargada correctamente.")
print("Preguntas registradas:")
for i, pregunta in enumerate(preguntas, 1):
    print(f"{i}. {pregunta}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 6.2 MB/s eta 0:00:00
API Key cargada correctamente.
Preguntas registradas:
1. ¿Cómo puedo mejorar la seguridad de una computadora personal?
2. ¿Qué estrategia recomendarías para estudiar matemáticas antes de un examen?
3. ¿Cómo puede una escuela utilizar inteligencia artificial de manera responsable?


In [2]:
# Definir la lista de preguntas
!pip install -q groq

from google.colab import userdata
from groq import Groq

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

client = Groq(api_key=GROQ_API_KEY)

print("✅ API Key encontrada")
print("✅ Cliente Groq creado correctamente")
print("✅ Ya puedes ejecutar el Reto 1")

✅ API Key encontrada
✅ Cliente Groq creado correctamente
✅ Ya puedes ejecutar el Reto 1


**2. Consulta la primera pregunta:** Envía la primera pregunta a Llama usando el modelo más ligero disponible en Groq. Guarda la respuesta, el tiempo de respuesta y el número de tokens en un diccionario llamado `resultado_1`.

In [3]:
# Consultar la primera pregunta y guardar el resultado en resultado_1
# Ejecutar después del Reto 1.
# RETO 2. Consultar la primera pregunta

import time

# Modelo actualizado de Groq
MODELO_LIGERO = "openai/gpt-oss-20b"

# Medir tiempo de respuesta
inicio = time.perf_counter()

respuesta = client.chat.completions.create(
    model=MODELO_LIGERO,
    messages=[
        {
            "role": "user",
            "content": preguntas[0]
        }
    ],
    temperature=0.2
)

fin = time.perf_counter()

# Calcular latencia
tiempo_respuesta = fin - inicio

# Guardar resultado
resultado_1 = {
    "pregunta": preguntas[0],
    "respuesta": respuesta.choices[0].message.content,
    "tiempo_segundos": round(tiempo_respuesta, 4),
    "tokens_prompt": respuesta.usage.prompt_tokens,
    "tokens_respuesta": respuesta.usage.completion_tokens,
    "tokens_totales": respuesta.usage.total_tokens,
    "modelo": MODELO_LIGERO
}

# Mostrar resultado
print("RESULTADO 1")
print("-------------------------")
print("Pregunta:")
print(resultado_1["pregunta"])

print("\nRespuesta:")
print(resultado_1["respuesta"])

print("\nDatos de ejecución:")
print(f"Tiempo: {resultado_1['tiempo_segundos']} segundos")
print(f"Tokens del prompt: {resultado_1['tokens_prompt']}")
print(f"Tokens de respuesta: {resultado_1['tokens_respuesta']}")
print(f"Tokens totales: {resultado_1['tokens_totales']}")
print(f"Modelo: {resultado_1['modelo']}")

RESULTADO 1
-------------------------
Pregunta:
¿Cómo puedo mejorar la seguridad de una computadora personal?

Respuesta:
## Guía práctica para reforzar la seguridad de tu PC personal

> **Objetivo**: Minimizar los riesgos de

Datos de ejecución:
Tiempo: 2.9136 segundos
Tokens del prompt: 81
Tokens de respuesta: 2048
Tokens totales: 2129
Modelo: openai/gpt-oss-20b


**3. Repite para la segunda y tercera pregunta:** Crea `resultado_2` y `resultado_3` de la misma forma.

In [4]:
# Consultar la segunda pregunta y guardar el resultado en resultado_2
# RETO 3.

import time

MODELO_LIGERO = "openai/gpt-oss-20b"

inicio = time.perf_counter()

respuesta = client.chat.completions.create(
    model=MODELO_LIGERO,
    messages=[
        {
            "role": "user",
            "content": preguntas[1]
        }
    ],
    temperature=0.2
)

fin = time.perf_counter()

resultado_2 = {
    "pregunta": preguntas[1],
    "respuesta": respuesta.choices[0].message.content,
    "tiempo_segundos": round(fin - inicio, 4),
    "tokens_prompt": respuesta.usage.prompt_tokens,
    "tokens_respuesta": respuesta.usage.completion_tokens,
    "tokens_totales": respuesta.usage.total_tokens,
    "modelo": MODELO_LIGERO
}

print("RESULTADO 2")
print("-------------------------")
print("Pregunta:")
print(resultado_2["pregunta"])

print("\nRespuesta:")
print(resultado_2["respuesta"])

print("\nDatos de ejecución:")
print(f"Tiempo: {resultado_2['tiempo_segundos']} segundos")
print(f"Tokens del prompt: {resultado_2['tokens_prompt']}")
print(f"Tokens de respuesta: {resultado_2['tokens_respuesta']}")
print(f"Tokens totales: {resultado_2['tokens_totales']}")
print(f"Modelo: {resultado_2['modelo']}")

RESULTADO 2
-------------------------
Pregunta:
¿Qué estrategia recomendarías para estudiar matemáticas antes de un examen?

Respuesta:
### Estrategia práctica para estudiar matemáticas antes de un examen

| Paso | Qué hacer | Por qué funciona | Tiempo recomendado |
|------|-----------|------------------|--------------------|
| **1. Revisa el temario y el formato del examen** | - Lee la guía del curso o el esquema de la prueba.<br>- Anota los temas obligatorios, los que se repiten y el tipo de preguntas (álgebra, geometría, cálculo, probabilidad, etc.). | Conoces exactamente qué esperar y no pierdes tiempo en lo que no se evalúa. | 15–20 min |
| **2. Haz un diagnóstico rápido** | - Resuelve 5–10 problemas de cada tema.<br>- Marca los que fallaste o te tomaron más de 2 min. | Identifica tus “puntos débiles” y enfoca tu energía donde más la necesitas. | 30 min |
| **3. Crea un plan de estudio** | - Divide el temario en bloques de 1–2 horas.<br>- Asigna más tiempo a los temas que fallaste

In [5]:
# Consultar la tercera pregunta y guardar el resultado en resultado_3

import time

MODELO_LIGERO = "openai/gpt-oss-20b"

inicio = time.perf_counter()

respuesta = client.chat.completions.create(
    model=MODELO_LIGERO,
    messages=[
        {
            "role": "user",
            "content": preguntas[2]
        }
    ],
    temperature=0.2
)

fin = time.perf_counter()

# Guardar el resultado de la tercera pregunta
resultado_3 = {
    "pregunta": preguntas[2],
    "respuesta": respuesta.choices[0].message.content,
    "tiempo_segundos": round(fin - inicio, 4),
    "tokens_prompt": respuesta.usage.prompt_tokens,
    "tokens_respuesta": respuesta.usage.completion_tokens,
    "tokens_totales": respuesta.usage.total_tokens,
    "modelo": MODELO_LIGERO
}

# Mostrar resultado
print("RESULTADO 3")
print("-------------------------")

print("Pregunta:")
print(resultado_3["pregunta"])

print("\nRespuesta:")
print(resultado_3["respuesta"])

print("\nDatos de ejecución:")
print(f"Tiempo: {resultado_3['tiempo_segundos']} segundos")
print(f"Tokens del prompt: {resultado_3['tokens_prompt']}")
print(f"Tokens de respuesta: {resultado_3['tokens_respuesta']}")
print(f"Tokens totales: {resultado_3['tokens_totales']}")
print(f"Modelo: {resultado_3['modelo']}")

RESULTADO 3
-------------------------
Pregunta:
¿Cómo puede una escuela utilizar inteligencia artificial de manera responsable?

Respuesta:
## Cómo una escuela puede usar la inteligencia artificial (IA) de manera responsable

La IA puede transformar la enseñanza y el aprendizaje, pero su adopción debe ir acompañada de una reflexión ética, legal y pedagógica. A continuación se presentan principios, pasos concretos y ejemplos prácticos que las escuelas pueden seguir para garantizar un uso responsable de la IA.

---

### 1. Principios rectores

| Principio | Qué significa | Por qué es importante |
|-----------|---------------|-----------------------|
| **Transparencia** | Los estudiantes, padres y docentes deben saber qué sistemas de IA se usan, cómo funcionan y qué datos procesan. | Fomenta la confianza y permite la supervisión. |
| **Equidad y no discriminación** | Los algoritmos deben evitar sesgos de género, raza, discapacidad, etc. | Evita reforzar desigualdades existentes. |
| **Pri

**4. Junta los resultados:** Agrega `resultado_1`, `resultado_2` y `resultado_3` a una lista vacía llamada `resultados`.

In [6]:
# Definir la lista resultados y agregar los tres diccionarios
# RETO 4. Consultar la tercera pregunta y reunir los tres resultados
# Ejecutar después del Reto 3.

import time

MODELO_LIGERO = "openai/gpt-oss-20b"

inicio = time.perf_counter()

respuesta = client.chat.completions.create(
    model=MODELO_LIGERO,
    messages=[
        {
            "role": "user",
            "content": preguntas[2]
        }
    ],
    temperature=0.2
)

fin = time.perf_counter()

resultado_3 = {
    "pregunta": preguntas[2],
    "respuesta": respuesta.choices[0].message.content,
    "tiempo_segundos": round(fin - inicio, 4),
    "tokens_prompt": respuesta.usage.prompt_tokens,
    "tokens_respuesta": respuesta.usage.completion_tokens,
    "tokens_totales": respuesta.usage.total_tokens,
    "modelo": MODELO_LIGERO
}

resultados = [resultado_1, resultado_2, resultado_3]

print("RESULTADO 3")
print("-------------------------")
print("Pregunta:")
print(resultado_3["pregunta"])

print("\nRespuesta:")
print(resultado_3["respuesta"])

print("\nDatos de ejecución:")
print(f"Tiempo: {resultado_3['tiempo_segundos']} segundos")
print(f"Tokens del prompt: {resultado_3['tokens_prompt']}")
print(f"Tokens de respuesta: {resultado_3['tokens_respuesta']}")
print(f"Tokens totales: {resultado_3['tokens_totales']}")
print(f"Modelo: {resultado_3['modelo']}")

print("\nLISTA DE RESULTADOS")
print(f"Se integraron {len(resultados)} resultados.")

RESULTADO 3
-------------------------
Pregunta:
¿Cómo puede una escuela utilizar inteligencia artificial de manera responsable?

Respuesta:
## Cómo una escuela puede utilizar inteligencia artificial (IA) de manera responsable

La IA puede transformar la educación: personalizar el aprendizaje, automatizar tareas administrativas, ofrecer nuevas formas de interacción y ampliar el acceso a recursos. Pero, como toda tecnología poderosa, también trae riesgos: sesgos, pérdida de privacidad, dependencia excesiva, falta de transparencia y brechas digitales.  

A continuación encontrarás una guía práctica, basada en principios éticos, normativas internacionales y buenas prácticas de instituciones educativas que ya están experimentando con IA de forma segura y efectiva.

---

### 1. Fundamentos éticos y marco normativo

| Principio | Qué significa | Por qué es importante |
|-----------|---------------|-----------------------|
| **Respeto a la dignidad humana** | La IA debe proteger la autonomía y

**5. Muestra la comparación:** Imprime `resultados` y concluye si el modelo ligero resolvió las 3 preguntas satisfactoriamente.

In [7]:
# Mostrar la tabla final de resultados
# RETO 5. Mostrar la comparación final y concluir si el modelo resolvió las preguntas

from IPython.display import display
import pandas as pd

if len(resultados) != 3:
    raise ValueError(
        "Se esperaban 3 resultados. Ejecuta correctamente los Retos 2, 3 y 4 antes del Reto 5."
    )

tabla = pd.DataFrame([
    {
        "Pregunta": i + 1,
        "Modelo": r["modelo"],
        "Tiempo (s)": r["tiempo_segundos"],
        "Tokens prompt": r["tokens_prompt"],
        "Tokens respuesta": r["tokens_respuesta"],
        "Tokens totales": r["tokens_totales"],
        "Respuesta": r["respuesta"]
    }
    for i, r in enumerate(resultados)
])

print("COMPARACIÓN FINAL")
print("=========================")

display(
    tabla[
        [
            "Pregunta",
            "Modelo",
            "Tiempo (s)",
            "Tokens prompt",
            "Tokens respuesta",
            "Tokens totales"
        ]
    ]
)

print("\nRESPUESTAS GENERADAS")
print("=========================")

for i, r in enumerate(resultados, 1):
    print(f"\n--- Pregunta {i} ---")
    print(r["pregunta"])
    print("\nRespuesta:")
    print(r["respuesta"])

resueltas = all(
    isinstance(r["respuesta"], str)
    and len(r["respuesta"].strip()) > 0
    for r in resultados
)

print("\nCONCLUSIÓN")
print("=========================")

if resueltas:
    print(
        "El modelo openai/gpt-oss-20b generó una respuesta para las 3 preguntas. "
        "La calidad de las respuestas debe valorarse considerando exactitud, pertinencia y utilidad."
    )
else:
    print("El modelo no generó correctamente las tres respuestas.")

print("\nLISTA COMPLETA DE RESULTADOS")
print("=========================")

for i, r in enumerate(resultados, 1):
    print(f"\nResultado {i}:")
    print(r)

COMPARACIÓN FINAL


,Pregunta,Modelo,Tiempo (s),Tokens prompt,Tokens respuesta,Tokens totales
0,1,openai/gpt-oss-20b,2.9136,81,2048,2129
1,2,openai/gpt-oss-20b,1.8340,84,1518,1602
2,3,openai/gpt-oss-20b,2.4205,82,2048,2130



RESPUESTAS GENERADAS

--- Pregunta 1 ---
¿Cómo puedo mejorar la seguridad de una computadora personal?

Respuesta:
## Guía práctica para reforzar la seguridad de tu PC personal

> **Objetivo**: Minimizar los riesgos de

--- Pregunta 2 ---
¿Qué estrategia recomendarías para estudiar matemáticas antes de un examen?

Respuesta:
### Estrategia práctica para estudiar matemáticas antes de un examen

| Paso | Qué hacer | Por qué funciona | Tiempo recomendado |
|------|-----------|------------------|--------------------|
| **1. Revisa el temario y el formato del examen** | - Lee la guía del curso o el esquema de la prueba.<br>- Anota los temas obligatorios, los que se repiten y el tipo de preguntas (álgebra, geometría, cálculo, probabilidad, etc.). | Conoces exactamente qué esperar y no pierdes tiempo en lo que no se evalúa. | 15–20 min |
| **2. Haz un diagnóstico rápido** | - Resuelve 5–10 problemas de cada tema.<br>- Marca los que fallaste o te tomaron más de 2 min. | Identifica tus “puntos